*Auto-generated from a Mathcad worksheet by mcad2py.*

In [ ]:
import math
import matplotlib.pyplot as plt
import pint

from mcad2py.runtime import power, elementwise, arange, double_integral, solve_block, sample, plot_axis, mesh_grid, resolve_plot_grid
ureg = pint.UnitRegistry()

In [ ]:
W = 600 * ureg.mm

In [ ]:
H = 800 * ureg.mm

In [ ]:
epsilon_0 = 0

In [ ]:
kappa_x = 1 * ureg.km**-1

In [ ]:
kappa_y = 3 * ureg.km**-1

In [ ]:
f_cd = 30 * ureg.MPa

In [ ]:
E_c = 33 * ureg.GPa

In [ ]:
f_ctd = 1 * ureg.MPa

In [ ]:
n = 2

In [ ]:
epsilon_c2 = -(2 / 1000)

In [ ]:
def sigma(e):
    if e > f_ctd / E_c:
        return f_ctd
    elif e > 0:
        return E_c * e
    elif e > epsilon_c2:
        return -f_cd * (1 - power(1 - e / epsilon_c2, n))
    return -f_cd
sigma = elementwise(sigma)

In [ ]:
e0 = arange(1.5 * epsilon_c2, -epsilon_c2, 1.49 * epsilon_c2 - 1.5 * epsilon_c2)

In [ ]:
_fig, _ax = plt.subplots()
_ax.plot(plot_axis(e0, 10**-3), plot_axis(sample(lambda e0: sigma(e0), e0), ureg.MPa), label='sigma(e0)', color='#00008B')
_ax.axhline(0, color='0.6', linewidth=0.8)
_ax.axvline(0, color='0.6', linewidth=0.8)
_ax.grid(True, alpha=0.3)
_ax.set_xlabel('e0 (10**-3)')
_ax.set_ylabel('(MPa)')
_ax.legend()
plt.show()

In [ ]:
N = lambda epsilon, kappa_x, kappa_y: double_integral(lambda x, y: sigma(epsilon + kappa_x * x + kappa_y * y), -W / 2, W / 2, -H / 2, H / 2)

In [ ]:
M_x = lambda epsilon, kappa_x, kappa_y: double_integral(lambda x, y: sigma(epsilon + kappa_x * x + kappa_y * y) * x, -W / 2, W / 2, -H / 2, H / 2)

In [ ]:
M_y = lambda epsilon, kappa_x, kappa_y: double_integral(lambda x, y: sigma(epsilon + kappa_x * x + kappa_y * y) * y, -W / 2, W / 2, -H / 2, H / 2)

In [ ]:
EA = H * W * E_c

In [ ]:
EIx = E_c * (1 / 12) * W**3 * H

In [ ]:
EIy = E_c * (1 / 12) * H**3 * W

In [ ]:
N_Ed = 0 * ureg.kN

In [ ]:
M_xEd = 0 * ureg.kN * ureg.m

In [ ]:
M_yEd = 155 * ureg.kN * ureg.m

In [ ]:
e = N_Ed / EA
kx = M_xEd / EIx
ky = M_yEd / EIy
def _residuals_epsilon_0_kappa_x_kappa_y(_x):
    e, kx, ky = _x
    return [
        N(e, kx, ky) - (N_Ed),
        M_x(e, kx, ky) - (M_xEd),
        M_y(e, kx, ky) - (M_yEd),
    ]
epsilon_0, kappa_x, kappa_y = solve_block(_residuals_epsilon_0_kappa_x_kappa_y, [e, kx, ky])

In [ ]:
epsilon = lambda x, y: epsilon_0 + kappa_x * x + kappa_y * y

In [ ]:
x0 = arange(-W / (2 * ureg.mm), W / (2 * ureg.mm), -0.495 * (W / ureg.mm) - -W / (2 * ureg.mm))

In [ ]:
y0 = arange(-H / (2 * ureg.mm), H / (2 * ureg.mm), -0.495 * (H / ureg.mm) - -H / (2 * ureg.mm))

In [ ]:
f = lambda x, y: sigma(epsilon(x * ureg.mm, y * ureg.mm))

In [ ]:
_X, _Y, _Z, _kind = resolve_plot_grid(mesh_grid(lambda x0, y0: f(x0, y0), x0, y0))
_Xs, _Ys, _Zs = plot_axis(_X), plot_axis(_Y), plot_axis(_Z, ureg.MPa)
_fig, _ax = plt.subplots()
if _kind == 'scatter':
    _cs = _ax.tricontourf(_Xs, _Ys, _Zs)
    _ax.tricontour(_Xs, _Ys, _Zs, colors='k', linewidths=0.5)
else:
    _cs = _ax.contourf(_Xs, _Ys, _Zs)
    _ax.contour(_Xs, _Ys, _Zs, colors='k', linewidths=0.5)
plt.colorbar(_cs, ax=_ax)
plt.show()